
# N10: Deep Architecture Ablation Study
**Objective:** Execute the ABLATION IMPERATIVE. This notebook evaluates three competing state-of-the-art methodologies side-by-side using the exact same 5-fold CV split to mathematically determine the absolute victor.

**Methodologies Tested:**
1. **Stochastic Seed-Blending (The Robust Baseline)**: N04 CatBoost averaged across 5 random seeds to squash variance.
2. **TabNet (SOTA Deep Tabular)**: Differentiable decision trees with sequential attention, tuned via Optuna.
3. **DAE Swap-Noise Embeddings**: An unsupervised PyTorch Autoencoder trained with 15% Swap-Noise to extract latent feature manifolds, stacked into CatBoost.


In [ ]:

!pip install -q pytorch-tabnet


In [ ]:

import pandas as pd
import numpy as np
import warnings
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
SEED = 2026
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Executing on: {DEVICE}")


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

# Feature Engineering
for df in [train_df, test_df]:
    df['sleep_bin'] = pd.qcut(df['sleep_duration'], q=5, duplicates='drop').astype(str)
    df['stress_sleep_interact'] = df['stress_level'].astype(str) + '_' + df['sleep_bin']

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])

X_train_raw = train_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)

num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(X_train_raw[num_cols])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


In [ ]:

# Helper: Target Encodings
def get_te_matrices(tr_i, va_i):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr = y_train[tr_i]
    
    fold_te_tr, fold_te_val = [], []
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    return X_tr_full, X_va_full


In [ ]:

# ABLATION A: Stochastic Seed-Blending
print("=== ABLATION A: STOCHASTIC SEED-BLENDING ===")
oof_A = np.zeros((len(train_df), 3))
N_SEEDS = 5

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr, X_va = get_te_matrices(tr_i, va_i)
    y_tr, y_va = y_train[tr_i], y_train[va_i]
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    
    fold_probs = np.zeros((len(va_i), 3))
    for s in range(N_SEEDS):
        m = CatBoostClassifier(iterations=800, learning_rate=0.03, depth=6, class_weights=weights, random_seed=SEED+s, verbose=0, task_type='GPU' if DEVICE=='cuda' else 'CPU')
        m.fit(X_tr, y_tr)
        fold_probs += m.predict_proba(X_va) / N_SEEDS
        
    oof_A[va_i] = fold_probs
    print(f"  Fold {fold+1} completed.")

score_A = balanced_accuracy_score(y_train, oof_A.argmax(axis=1))
print(f"--> Ablation A (Seed Blend) Score: {score_A:.5f}\n")


In [ ]:

# ABLATION B: TabNet with Optuna
print("=== ABLATION B: TABNET ===")
oof_B = np.zeros((len(train_df), 3))

# For ablation speed, we tune hyperparameters on Fold 1 only, then use them for all folds.
tr_i_0, va_i_0 = next(skf.split(X_train_raw, y_train))
X_tr_0, X_va_0 = get_te_matrices(tr_i_0, va_i_0)

scaler = StandardScaler()
X_tr_0_s = scaler.fit_transform(X_tr_0)
X_va_0_s = scaler.transform(X_va_0)

def objective(trial):
    n_d = trial.suggest_int('n_d', 8, 32)
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    lambda_sparse = trial.suggest_float('lambda_sparse', 1e-4, 1e-2, log=True)
    
    clf = TabNetClassifier(n_d=n_d, n_a=n_d, n_steps=3, gamma=gamma, lambda_sparse=lambda_sparse, optimizer_params=dict(lr=2e-2), verbose=0)
    clf.fit(X_tr_0_s, y_train[tr_i_0], eval_set=[(X_va_0_s, y_train[va_i_0])], patience=10, max_epochs=50)
    preds = clf.predict(X_va_0_s)
    return balanced_accuracy_score(y_train[va_i_0], preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)
best_p = study.best_params

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr, X_va = get_te_matrices(tr_i, va_i)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    
    clf = TabNetClassifier(n_d=best_p['n_d'], n_a=best_p['n_d'], n_steps=3, gamma=best_p['gamma'], lambda_sparse=best_p['lambda_sparse'], optimizer_params=dict(lr=2e-2), verbose=0)
    clf.fit(X_tr_s, y_train[tr_i], eval_set=[(X_va_s, y_train[va_i])], patience=10, max_epochs=50)
    oof_B[va_i] = clf.predict_proba(X_va_s)
    print(f"  Fold {fold+1} completed.")

score_B = balanced_accuracy_score(y_train, oof_B.argmax(axis=1))
print(f"--> Ablation B (TabNet) Score: {score_B:.5f}\n")


In [ ]:

# ABLATION C: DAE Swap-Noise Embeddings
print("=== ABLATION C: DAE SWAP-NOISE ===")
oof_C = np.zeros((len(train_df), 3))

class SwapNoiseDataset(Dataset):
    def __init__(self, X, swap_prob=0.15):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.swap_prob = swap_prob
        self.n_cols = X.shape[1]
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        mask = torch.rand(self.n_cols) < self.swap_prob
        swap_idx = torch.randint(0, len(self.X), (self.n_cols,))
        for c in range(self.n_cols):
            if mask[c]: x[c] = self.X[swap_idx[c], c]
        return x, self.X[idx]

class DAE(nn.Module):
    def __init__(self, in_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 64), nn.ReLU(), nn.Linear(64, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, in_dim))
    def forward(self, x):
        return self.decoder(self.encoder(x))

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr, X_va = get_te_matrices(tr_i, va_i)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    
    # Train DAE
    dae = DAE(in_dim=X_tr_s.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(dae.parameters(), lr=1e-3)
    loader = DataLoader(SwapNoiseDataset(X_tr_s), batch_size=512, shuffle=True)
    
    dae.train()
    for ep in range(15):
        for noisy_x, clean_x in loader:
            noisy_x, clean_x = noisy_x.to(DEVICE), clean_x.to(DEVICE)
            opt.zero_grad()
            loss = nn.MSELoss()(dae(noisy_x), clean_x)
            loss.backward()
            opt.step()
            
    dae.eval()
    with torch.no_grad():
        emb_tr = dae.encoder(torch.tensor(X_tr_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        emb_va = dae.encoder(torch.tensor(X_va_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        
    X_tr_aug = np.hstack([X_tr, emb_tr])
    X_va_aug = np.hstack([X_va, emb_va])
    
    class_counts = np.bincount(y_train[tr_i])
    weights = [len(y_train[tr_i]) / (len(class_counts) * c) for c in class_counts]
    m = CatBoostClassifier(iterations=800, learning_rate=0.03, depth=6, class_weights=weights, random_seed=SEED+fold, verbose=0, task_type='GPU' if DEVICE=='cuda' else 'CPU')
    m.fit(X_tr_aug, y_train[tr_i])
    oof_C[va_i] = m.predict_proba(X_va_aug)
    print(f"  Fold {fold+1} completed.")

score_C = balanced_accuracy_score(y_train, oof_C.argmax(axis=1))
print(f"--> Ablation C (DAE Swap-Noise) Score: {score_C:.5f}\n")


In [ ]:

print("=== FINAL ABLATION RESULTS ===")
print(f"Stochastic Seed-Blending : {score_A:.5f}")
print(f"TabNet (Optuna Tuned)    : {score_B:.5f}")
print(f"DAE Swap-Noise Embeddings: {score_C:.5f}")
